<a href="https://colab.research.google.com/github/DrewThomasson/audio.cpp/blob/main/Notebooks/colab_audio_cpp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# audio.cpp WebUI
Run the cell below to build audio.cpp with CUDA and open its WebUI. The public link remains active while the cell is running. Anyone with the link can use the WebUI, so do not share it.

In [ ]:
# @title Start audio.cpp

import os
import platform
import re
import subprocess
import time
import urllib.request
from pathlib import Path

from IPython.display import HTML, clear_output, display

REPO_DIR = Path("/content/audio.cpp")
REPO_URL = "https://github.com/DrewThomasson/audio.cpp.git"
REPO_BRANCH = "main"
BUILD_DIR = REPO_DIR / "build/colab-cuda"
SERVER_PORT = 8080
BUILD_LOG = Path("/content/audio_cpp_build.log")
SERVER_LOG = Path("/content/audio_cpp_server.log")
TUNNEL_LOG = Path("/content/audio_cpp_tunnel.log")
CLOUDFLARED = Path("/content/cloudflared")

def run(command, *, cwd=None, env=None, log=None):
    if log is None:
        subprocess.run(command, cwd=cwd, env=env, check=True)
        return
    with log.open("a", encoding="utf-8") as output:
        try:
            subprocess.run(
                command, cwd=cwd, env=env, check=True, stdout=output, stderr=subprocess.STDOUT
            )
        except subprocess.CalledProcessError as error:
            command_text = " ".join(map(str, command))
            raise RuntimeError(f"Command failed: {command_text}\n\n{tail(log)}") from error

def tail(path, line_count=40):
    if not path.exists():
        return ""
    return "".join(path.read_text(encoding="utf-8", errors="replace").splitlines(True)[-line_count:])

def stop(process):
    if process is None or process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()

if subprocess.run(
    ["nvidia-smi"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
).returncode != 0:
    raise RuntimeError(
        "No NVIDIA GPU was detected. In Colab, select Runtime > Change runtime type > T4 GPU, "
        "restart the session, and run this cell again."
    )

if platform.machine() != "x86_64":
    raise RuntimeError(f"This notebook currently supports x86_64 Colab runtimes, not {platform.machine()}.")

BUILD_LOG.write_text("", encoding="utf-8")
SERVER_LOG.write_text("", encoding="utf-8")
TUNNEL_LOG.write_text("", encoding="utf-8")

print("Preparing the Colab runtime...", flush=True)
run(["apt-get", "update", "-qq"], log=BUILD_LOG)
run(
    [
        "apt-get", "install", "-y", "-qq",
        "build-essential", "ca-certificates", "cmake", "curl", "ffmpeg",
        "libssl-dev", "ninja-build", "software-properties-common",
    ],
    log=BUILD_LOG,
)
if subprocess.run(["bash", "-lc", "command -v gcc-13 && command -v g++-13"], stdout=subprocess.DEVNULL).returncode != 0:
    run(["add-apt-repository", "-y", "ppa:ubuntu-toolchain-r/test"], log=BUILD_LOG)
    run(["apt-get", "update", "-qq"], log=BUILD_LOG)
run(["apt-get", "install", "-y", "-qq", "gcc-13", "g++-13"], log=BUILD_LOG)

if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout. Restart the runtime and try again.")
    print("Downloading audio.cpp...", flush=True)
    run(["git", "clone", "--depth=1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], log=BUILD_LOG)
else:
    print("Using the existing audio.cpp checkout.", flush=True)

compute_capability = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
).splitlines()[0].strip()
cuda_arch = compute_capability.replace(".", "")
if not cuda_arch.isdigit():
    raise RuntimeError(f"Could not determine the CUDA architecture from: {compute_capability!r}")

build_env = os.environ.copy()
build_env.update({"CC": "gcc-13", "CXX": "g++-13", "CUDAHOSTCXX": "g++-13"})
jobs = max(1, min(os.cpu_count() or 2, 4))
server_binary = BUILD_DIR / "bin/audiocpp_server"

if not server_binary.exists():
    print("Building audio.cpp with CUDA. This can take several minutes...", flush=True)
    run(
        [
            "bash", "scripts/build_linux.sh",
            "--backend", "cuda",
            "--build-dir", str(BUILD_DIR),
            "--build-type", "Release",
            "--cuda-arch", cuda_arch,
            "--deployment-build",
            "--native-model-manager",
            "--system-openssl",
            "--target", "audiocpp_server",
            "--jobs", str(jobs),
        ],
        cwd=REPO_DIR,
        env=build_env,
        log=BUILD_LOG,
    )
else:
    print("Using the existing audio.cpp build.", flush=True)

if not CLOUDFLARED.exists():
    print("Installing the public tunnel...", flush=True)
    run(
        [
            "curl", "-L", "--fail", "--silent", "--show-error",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-o", str(CLOUDFLARED),
        ],
        log=BUILD_LOG,
    )
    CLOUDFLARED.chmod(0o755)

server_process = None
tunnel_process = None
server_output = None
tunnel_output = None

try:
    print("Starting the audio.cpp WebUI...", flush=True)
    server_output = SERVER_LOG.open("w", encoding="utf-8")
    server_process = subprocess.Popen(
        [
            str(server_binary), "--ui", "--ui-management",
            "--backend", "cuda", "--host", "127.0.0.1",
            "--port", str(SERVER_PORT),
        ],
        cwd=REPO_DIR,
        stdout=server_output,
        stderr=subprocess.STDOUT,
        text=True,
    )

    health_url = f"http://127.0.0.1:{SERVER_PORT}/health"
    deadline = time.monotonic() + 120
    while time.monotonic() < deadline:
        if server_process.poll() is not None:
            raise RuntimeError("audio.cpp stopped during startup.\n\n" + tail(SERVER_LOG))
        try:
            with urllib.request.urlopen(health_url, timeout=2) as response:
                if response.status == 200:
                    break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("audio.cpp did not become ready.\n\n" + tail(SERVER_LOG))

    tunnel_output = TUNNEL_LOG.open("w", encoding="utf-8")
    tunnel_process = subprocess.Popen(
        [str(CLOUDFLARED), "tunnel", "--url", f"http://127.0.0.1:{SERVER_PORT}", "--no-autoupdate"],
        stdout=tunnel_output,
        stderr=subprocess.STDOUT,
        text=True,
    )

    public_url = None
    deadline = time.monotonic() + 90
    while time.monotonic() < deadline:
        if tunnel_process.poll() is not None:
            raise RuntimeError("The public tunnel stopped during startup.\n\n" + tail(TUNNEL_LOG))
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", TUNNEL_LOG.read_text(encoding="utf-8", errors="replace"))
        if match:
            public_url = match.group(0)
            break
        time.sleep(1)
    if public_url is None:
        raise RuntimeError("The public tunnel did not provide a URL.\n\n" + tail(TUNNEL_LOG))

    clear_output(wait=True)
    display(HTML(
        f'<h2>audio.cpp is running</h2>'
        f'<p><a href="{public_url}" target="_blank" rel="noopener noreferrer">Open the audio.cpp WebUI</a></p>'
        '<p>Keep this cell running while you use the WebUI. Anyone with the link can access this session.</p>'
    ))

    while True:
        if server_process.poll() is not None:
            raise RuntimeError("audio.cpp stopped unexpectedly.\n\n" + tail(SERVER_LOG))
        if tunnel_process.poll() is not None:
            raise RuntimeError("The public tunnel stopped unexpectedly.\n\n" + tail(TUNNEL_LOG))
        time.sleep(5)
except KeyboardInterrupt:
    clear_output(wait=True)
    print("audio.cpp stopped.")
finally:
    stop(tunnel_process)
    stop(server_process)
    if tunnel_output is not None:
        tunnel_output.close()
    if server_output is not None:
        server_output.close()
